# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [16]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping


DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  11.0 #/s median value from BRENDA
DEFAULT_KCAT_TRANSPORT = 100000000 #/s

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_ecolik12_240726.xlsx')

GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_eco_230309.json')

In [17]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iML1515_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'{formatted_date}_eco_ActiveEnzymeInformation_GotEnzymes.xlsx')

In [18]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'\b([b|s]\d{4})\b'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BiGG database](http://bigg.ucsd.edu/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iML1515 model for [*Escherichia coli* K12 substr. MG1655](http://bigg.ucsd.edu/models/iML1515)
- Important: the identifiers must me mappable to uniprot (or another enzyme identifier database) and KEGG (for mapping to GotEnzymes)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `eco` (*E. coli* strain K12) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *E. coli* we used [this](https://www.uniprot.org/uniprotkb?query=%28organism_id%3A83333%29) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [51]:
#load the model
model = read_sbml_model(os.path.join('Models', 'iML1515.xml'))

#make a id mapping and create a mapping dataframe
id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'rhea_id', 'mnx_id','Reactants','Products','EC', 'GPR'])
for rxn in model.reactions:
    #skip transport reactions and ATP maintenance requirement
    if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
        continue
    to_append= []   
    to_append.append(rxn.annotation['bigg.reaction'])
    #not all reactions are annotated with a KEGG ID in the model
    try: 
        to_append.append(rxn.annotation['kegg.reaction'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['rhea'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['metanetx.reaction'])
    except:
        to_append.append(np.nan)
    try:
        to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
    except:
        to_append.append([])
    try: 
        to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
    except:
        to_append.append([])
    if 'ec-code' in rxn.annotation.keys():
        to_append.append(rxn.annotation['ec-code'])
        to_append.append(rxn.gene_reaction_rule)  

    else:
        to_append.append(np.nan)
        to_append.append(rxn.gene_reaction_rule)
    id_mapper_df.loc[len(id_mapper_df)] = to_append



print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
print('Number of reactions mapped to rhea identifier: ', len(id_mapper_df.rhea_id.dropna()))
print('Number of reactions mapped to MetaNetX identifier: ', len(id_mapper_df.mnx_id.dropna()))
id_mapper_df.head()

DHORD5: dhor__S_c + mqn8_c --> mql8_c + orot_c
DHORD2: dhor__S_c + q8_c --> orot_c + q8h2_c
DHORDfum: dhor__S_c + fum_c --> orot_c + succ_c
Number of reactions in the mapping dataframe:  2028
Number of reactions mapped to KEGG identifier:  742
Number of reactions mapped to rhea identifier:  978
Number of reactions mapped to MetaNetX identifier:  1902


,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125
3,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518
4,SHK3Dr,R02413,"[17739, 17738, 17737, 17740]",MNXR104378,"[C02637, C00080, C00005]","[C00006, C00493]","[1.1.1.25, 1.1.1.282]",b3281 or b1692


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 2.1 Parse the BIGG gene-protein-reaction associations

In [53]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    return re.findall(locustag_regex, text)

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066,b2066
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238,b0238
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0238
3,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0125
4,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518,b0474
...,...,...,...,...,...,...,...,...,...
3611,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857
3612,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856
3613,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857
3614,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001


### 2.2 Parse the Uniprot data for merging

In [61]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
# fill in the missing genes to prevent a mismatch in mapping to the model data
uniprot_df['b_number'] = uniprot_df['b_number'].fillna('missing')
uniprot_df

/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Entry,Entry Name,Length,Mass,Rhea ID,b_number
0,A5A616,MGTS_ECOLI,31,3509,NaN,b4599
1,O32583,THIS_ECOLI,66,7311,NaN,b4407
2,P00350,6PGD_ECOLI,468,51481,RHEA:10116,b2029
3,P00363,FRDA_ECOLI,602,65972,RHEA:40523 RHEA:27834,b4154
4,P00370,DHE4_ECOLI,447,48581,RHEA:11612,b1761
...,...,...,...,...,...,...
4582,Q9S4X4,YUBB_ECOLI,145,16507,NaN,missing
4583,Q9S4X5,YUBA_ECOLI,168,19067,NaN,missing
4584,Q9XB42,YKFH_ECOLI,73,8581,NaN,b4504
4585,Q9Z3A0,YJGW_ECOLI,111,13085,NaN,b4274


### 2.3 Match Uniprot to BIGG data

In [62]:

id_mapper_with_protein = pd.merge(id_mapper_df, uniprot_df, on='b_number', how='left')

id_mapper_with_protein

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,Entry,Entry Name,Length,Mass,Rhea ID
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
3,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0125,P0A9M2,HPRT_ECOLI,178.0,20115.0,RHEA:17973 RHEA:25424
4,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518,b0474,P69441,KAD_ECOLI,214.0,23586.0,RHEA:12973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3611,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
3612,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856,P32125,MOBB_ECOLI,175.0,19363.0,NaN
3613,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
3614,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001,NaN,NaN,NaN,NaN,NaN


## 3. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [63]:
id_mapper_final = id_mapper_with_protein
id_mapper_final = id_mapper_final.rename({'Entry':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,enzyme_id,Entry Name,Length,Mass,Rhea ID
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.48,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
1,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.213,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
2,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]",2.4.2.8,b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
3,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]",2.4.2.22,b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
4,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973


In [66]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
# eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,b0002,R01773,1.1.1.3,C00263,2.1410
1,b0002,R01773,1.1.1.3,C00441,5.0932
2,b0002,R01775,1.1.1.3,C00263,2.1410
3,b0002,R01775,1.1.1.3,C00441,5.0932
4,b0002,R00480,2.7.2.4,C03082,5.0319


In [67]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,Entry Name,Length,Mass,Rhea ID,ec_number,compound,kcat_values
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.48,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,2.7.1.48,C00055,162.9332
1,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.48,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,2.7.1.48,C00475,150.7242
2,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.213,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,2.7.1.48,C00055,162.9332
3,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.213,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674,2.7.1.48,C00475,150.7242
4,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]",2.4.2.8,b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973,2.4.2.8,C00655,0.0056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9234,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243,NaN,NaN,NaN
9235,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856,P32125,MOBB_ECOLI,175.0,19363.0,NaN,NaN,NaN,NaN
9236,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243,NaN,NaN,NaN
9237,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [68]:
# Set default values
eco_enzymes_parsed = eco_enzymes_merged.copy()
eco_enzymes_parsed['direction'] = 'f'
eco_enzymes_parsed['kcat_values'] = eco_enzymes_parsed['kcat_values'].fillna(DEFAULT_KCAT)
eco_enzymes_parsed['Mass'] = eco_enzymes_parsed['Mass'].fillna(DEFAULT_PROT_MW)


# Determine direction of the reaction by determining whether the kcat is assigned to the enzyme-substrate or enzyme-product pain
eco_enzymes_parsed['direction'] = np.where(
    eco_enzymes_parsed['compound'].isin(eco_enzymes_parsed['Products']), 'b',
    np.where(eco_enzymes_parsed['compound'].isin(eco_enzymes_parsed['Reactants']), 
             'f', 
             eco_enzymes_parsed['direction'])
)

# Handle missing Mass and unmappable proteins with a GPR annotation
eco_enzymes_parsed['Mass'] = np.where(
    (eco_enzymes_parsed['Mass'].isna()) & (eco_enzymes_parsed['GPR'].str.len() > 2),
    DEFAULT_PROT_MW,
    eco_enzymes_parsed['Mass']
)

# Assign genes if missing
eco_enzymes_parsed['gene'] = eco_enzymes_parsed.apply(
    lambda row: f'gene_{row.enzyme_id}' if isinstance(row.gene, float) else row.gene, axis=1
)

# Handle specific case for 's0001' (annotation is gene is not mapped to a genome annotation)
eco_enzymes_parsed.loc[eco_enzymes_parsed['GPR'] == 's0001', 'gene'] = eco_enzymes_parsed['GPR']
eco_enzymes_parsed.loc[eco_enzymes_parsed['GPR'] == 's0001', 'enzyme_id'] = 'Enzyme_s0001' + eco_enzymes_parsed['rxn_id']


# Clean up final columns
eco_enzymes_parsed = eco_enzymes_parsed.drop(['ec_number', 'compound'], axis=1).rename(columns={'Mass': 'molMass'})
eco_enzymes_parsed

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,Entry Name,Length,molMass,Rhea ID,kcat_values,direction
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.48,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,162.9332,f
1,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.48,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,150.7242,f
2,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.213,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,162.9332,f
3,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]",2.7.1.213,b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0000,RHEA:16825 RHEA:24674,150.7242,f
4,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]",2.4.2.8,b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0000,RHEA:25424 RHEA:10800 RHEA:17973,0.0056,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9234,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0000,RHEA:34243,11.0000,f
9235,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856,P32125,MOBB_ECOLI,175.0,19363.0000,NaN,11.0000,f
9236,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0000,RHEA:34243,11.0000,f
9237,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001,Enzyme_s0001FESD2s,NaN,NaN,39959.4825,NaN,11.0000,f


In [75]:
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_parsed, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_parsed, gene2protein)
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(['rxn_id', 'enzyme_id', 'direction'])

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,Entry Name,Length,molMass,Rhea ID,kcat_values,direction
7883,12DGR120tipp,NaN,NaN,MNXR94675,[],[],NaN,,gene_nan,NaN,NaN,NaN,39959.4825,NaN,11.0,f
7847,12DGR140tipp,NaN,NaN,MNXR94676,[],[],NaN,,gene_nan,NaN,NaN,NaN,39959.4825,NaN,11.0,f
7289,12DGR141tipp,NaN,NaN,MNXR94677,[],[],NaN,,gene_nan,NaN,NaN,NaN,39959.4825,NaN,11.0,f
7849,12DGR160tipp,NaN,NaN,MNXR94678,[],[],NaN,,gene_nan,NaN,NaN,NaN,39959.4825,NaN,11.0,f
8122,12DGR161tipp,NaN,NaN,MNXR94679,[],[],NaN,,gene_nan,NaN,NaN,NaN,39959.4825,NaN,11.0,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6146,ZN2abcpp,NaN,"[20622, 20624, 20623, 20621]",MNXR105277,"[C00002, [C00001, C01328], C00038]","[C00008, C00080, C00009, C00038]",3.6.3.5,b3469,[b3469],P37617,ZNTA_ECOLI,732.0,76840.0000,RHEA:52580 RHEA:20621 RHEA:12132,11.0,f
6845,ZN2t3pp,NaN,"[28842, 28840, 28841, 28839]",MNXR105278,"[C00080, C00038]","[C00080, C00038]",NaN,b3915 or b0752,[b3915],P69380,FIEF_ECOLI,300.0,32927.0000,RHEA:28839 RHEA:28739 RHEA:29439,11.0,f
6846,ZN2t3pp,NaN,"[28842, 28840, 28841, 28839]",MNXR105278,"[C00080, C00038]","[C00080, C00038]",NaN,b3915 or b0752,[b0752],P75757,ZITB_ECOLI,313.0,34678.0000,NaN,11.0,f
8149,ZN2tpp,NaN,"[29353, 29351, 29352, 29354]",MNXR105280,[C00038],[C00038],NaN,b3040,[b3040],P0A8H3,ZUPT_ECOLI,257.0,26485.0000,RHEA:29351 RHEA:28707 RHEA:28578 RHEA:28486 RH...,11.0,f


In [77]:
#drop duplicate entries. If there are duplicates, make sure the mean kcat value is used to parametrize
eco_enzymes_final = eco_enzymes_mapped.groupby(
    ['rxn_id', 'enzyme_id', 'direction'], 
    as_index=False
).agg({
    'kcat_values': 'mean', 
    **{col: 'first' for col in eco_enzymes_mapped.columns if col not in ['rxn_id', 'enzyme_id', 'direction', 'kcat']
      }
})
# eco_enzymes_final.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [78]:
# Get the information about the enzyme sectors
other_sectors = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = None)
del other_sectors['ActiveEnzymes']

In [79]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes')
    for sheet, df in other_sectors.items():
        if sheet == 'ExcessEnzymes': sheet = 'UnusedEnzyme'
        df.to_excel(writer, sheet_name=sheet)